In [1]:
from collections import defaultdict
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader,random_split
import torch.optim as optim
random.seed(42)

In [2]:
if torch.cuda.is_available():
    print("cuda is available")

cuda is available


In [3]:
train1 = torch.load("./train_val_embedding_data/circuit_breakers_others.pt", map_location=torch.device('cpu'))
train2 = torch.load("./train_val_embedding_data/circuit_breakers_actorattack.pt", map_location=torch.device('cpu'))

C:\Users\Karim Mahmoud\AppData\Local\Temp\ipykernel_13884\405173482.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  train1 = torch.load("./train_val_embedding_data/circu

In [4]:
train1[0][0].keys()

dict_keys(['u', 'y', 'score', 'round'])

In [5]:
scores1=defaultdict(int)
scores2=defaultdict(int)
for conversation in train1:
    for turn in conversation:
        scores1[turn['score']]+=1

for conversation in train2:
    for turn in conversation:
        scores2[turn['score']]+=1

print("Score distribution:")
print(sorted(scores1.items()))
print(sorted(scores2.items()))

Score distribution:
[(1, 4629), (2, 2442), (3, 6199), (4, 1206), (5, 1724)]
[(1, 2719), (2, 2170), (3, 4848), (4, 2149), (5, 893)]


In [6]:
convo1_len=defaultdict(int)
convo2_len=defaultdict(int)
for conversation in train1:
    convo1_len[len(conversation)]+=1
for conversation in train2:
    convo2_len[len(conversation)]+=1

print("Conversation length distribution:")
print(sorted(convo1_len.items()))
print(sorted(convo2_len.items()))

Conversation length distribution:
[(1, 583), (2, 151), (3, 179), (4, 226), (5, 201), (6, 134), (7, 143), (8, 1383)]
[(1, 6), (2, 5), (3, 4), (4, 3), (5, 1115), (6, 1194)]


In [7]:
original_training_data=train1+train2
training_data = random.sample(original_training_data, len(original_training_data))

assert len(training_data)==len(train1)+len(train2)
assert training_data!=original_training_data
assert len(training_data)==len(original_training_data)

# dataset_prep

In [8]:
class multi_turn_dataset(Dataset):
    def __init__(self, conversations):
        self.conversations = conversations
        self.longest_convo = max(len(conv) for conv in conversations)

    def __len__(self):
        return len(self.conversations)

    def __getitem__(self, idx):
        convo = self.conversations[idx]
        convo_len = len(convo) 
        user_inputs=torch.stack([turn['u'] for turn in convo])
        ai_responses=torch.stack([turn['y'] for turn in convo])
        scores=torch.tensor([turn['score'] for turn in convo])
        pad_size = self.longest_convo - convo_len
        mask=torch.ones(convo_len)

        if pad_size > 0:
            u_pad = torch.zeros((pad_size, *user_inputs.shape[1:]))
            y_pad = torch.zeros((pad_size, *ai_responses.shape[1:]))
            s_pad = torch.zeros(pad_size)

            user_inputs = torch.cat([user_inputs, u_pad], dim=0)
            ai_responses = torch.cat([ai_responses, y_pad], dim=0)
            scores = torch.cat([scores, s_pad], dim=0)
            mask=torch.cat([mask, torch.zeros(pad_size)], dim=0)
        
        return user_inputs, ai_responses, scores, mask

In [9]:
def split_data(dataset,batch_size, validation_split=0.05):
    dataset_size = len(dataset)
    val_size = int(validation_split * dataset_size)
    train_size = dataset_size - val_size
    
    train_dataset, val_dataset = random_split(
        dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)
    return train_loader, val_loader

In [10]:
dataset=multi_turn_dataset(training_data)
train_loader, val_loader = split_data(dataset, batch_size=32)

In [11]:
u,v,scores,mask=next(iter(train_loader))

In [18]:
u.shape

torch.Size([32, 8, 768])

In [15]:
mask.shape

torch.Size([32, 8])

In [17]:
scores.shape

torch.Size([32, 8])

In [16]:
mask.unsqueeze(-1).shape

torch.Size([32, 8, 1])

# Modeling

In [13]:
class stateSpaceModel(nn.Module):
    def __init__(self,state_dim, input_dim, hidden_dim1,hidden_dim2, output_dim):
        super(stateSpaceModel, self).__init__()
        # F(x_{t-1},u_t)=>x_t (Transition model)
        self.Fxu = nn.Sequential(
            nn.Linear(state_dim + input_dim, hidden_dim1),
            nn.ReLU(),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Linear(hidden_dim2, state_dim)
        )

        # G(x_t,u_t)=>z_t (Observation model)
        self.Gxu = nn.Sequential(
            nn.Linear(state_dim + input_dim, hidden_dim1),
            nn.ReLU(),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Linear(hidden_dim2, output_dim)
        )
        
    def forward(self, x_prev, u):
        # gets xt
        ux_prev = torch.cat([u,x_prev], dim=-1)
        x_curr = self.Fxu(ux_prev)

        # gets zt
        ux_curr = torch.cat([u,x_curr], dim=-1)
        zt = self.Gxu(ux_curr)
        return x_curr, zt

In [ ]:
state_dim = 768    
input_dim = 768    
output_dim = 768   
hidden_dim_ssm1 = 1200  
hidden_dim_ssm2 = 900  

state_model=stateSpaceModel(state_dim=state_dim, input_dim=input_dim, hidden_dim1=hidden_dim_ssm1, hidden_dim2=hidden_dim_ssm2, output_dim=output_dim)

In [15]:
print(state_model)

stateSpaceModel(
  (Fxu): Sequential(
    (0): Linear(in_features=1536, out_features=1200, bias=True)
    (1): ReLU()
    (2): Linear(in_features=1200, out_features=900, bias=True)
    (3): ReLU()
    (4): Linear(in_features=900, out_features=768, bias=True)
  )
  (Gxu): Sequential(
    (0): Linear(in_features=1536, out_features=1200, bias=True)
    (1): ReLU()
    (2): Linear(in_features=1200, out_features=900, bias=True)
    (3): ReLU()
    (4): Linear(in_features=900, out_features=768, bias=True)
  )
)


# train

In [16]:
def train_dialogue_dynamics(state_model,train_loader,val_loader, num_epochs=200, save_path="ssm_model.pth", weight_decay=0,ssm_learning_rate=1e-4,state_dim=332): 
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    state_model.to(device)
    optimizer_ssm = optim.Adam(state_model.parameters(), lr=ssm_learning_rate, weight_decay=weight_decay)

    loss_fn_mse = nn.MSELoss()    
    best_val_loss_ssm = float('inf')  
    for epoch in range(num_epochs):
        # Training
        state_model.train()
        total_ssm_loss = 0.0

        for u_batch, z_batch, _ , mask in train_loader:
            u_batch = u_batch.to(device)
            z_batch = z_batch.to(device)
            mask = mask.unsqueeze(-1).to(device)
            batch_size, seq_len, state_dim = u_batch.shape
            # state_dim = state_dim
            x_t = torch.zeros(batch_size, state_dim, device=device)
            predicted_z = []
            # loop on convo turns
            for t in range(seq_len):
                u_t = u_batch[:, t, :]
                x_t, z_t = state_model(x_t, u_t)
                predicted_z.append(z_t)
            
            predicted_z = torch.stack(predicted_z, dim=1)

            ssm_loss = loss_fn_mse(predicted_z * mask, z_batch * mask)
            optimizer_ssm.zero_grad()
            ssm_loss.backward() 
            optimizer_ssm.step()

            total_ssm_loss += ssm_loss.item()
        
        # Validation
        state_model.eval()
        val_total_ssm_loss = 0.0
        with torch.no_grad():
            for u_batch, z_batch, _ , mask in val_loader:
                u_batch = u_batch.to(device)
                z_batch = z_batch.to(device)
                mask = mask.unsqueeze(-1).to(device)
                batch_size, seq_len, state_dim = u_batch.shape

                x_t = torch.zeros(batch_size, state_dim, device=device)

                predicted_z = []
                for t in range(seq_len):
                    u_t = u_batch[:, t, :]
                    x_t, z_t = state_model(x_t, u_t)
                    predicted_z.append(z_t)

                predicted_z = torch.stack(predicted_z, dim=1)

                ssm_loss = loss_fn_mse(predicted_z * mask, z_batch * mask)

                val_total_ssm_loss += ssm_loss.item()
                
        # Average losses
        avg_train_ssm_loss = total_ssm_loss / len(train_loader)
        avg_val_total_ssm_loss = val_total_ssm_loss / len(val_loader)


        print(f"Epoch {epoch + 1}/{num_epochs}"
              f"Train SSM Loss: {avg_train_ssm_loss}"
              f"Val SSM Loss: {avg_val_total_ssm_loss}")

        if avg_val_total_ssm_loss < best_val_loss_ssm:
            best_val_loss_ssm = avg_val_total_ssm_loss
            torch.save({
                'ssm': state_model.state_dict(),
            }, f"{save_path}/models_best_ssm.pth")
            print(f"Models saved as models_best_ssm, ssm loss: {avg_val_total_ssm_loss}")
        
        if (epoch + 1) % 30 == 0:
            torch.save({
                'ssm': state_model.state_dict(),
            }, save_path+  f"/ssm_model_epoch{epoch + 1}.pth")

In [17]:
train_dialogue_dynamics(state_model, train_loader, val_loader, num_epochs=200, save_path="./ssm_model_exp5", weight_decay=0, ssm_learning_rate=1e-4)

Epoch 1/200Train SSM Loss: 0.0007194001619329196Val SSM Loss: 0.0005553429105526044
Models saved as models_best_ssm, ssm loss: 0.0005553429105526044
Epoch 2/200Train SSM Loss: 0.0005321906535057409Val SSM Loss: 0.0004661761502373136
Models saved as models_best_ssm, ssm loss: 0.0004661761502373136
Epoch 3/200Train SSM Loss: 0.00046524285146990006Val SSM Loss: 0.00042339949221867655
Models saved as models_best_ssm, ssm loss: 0.00042339949221867655
Epoch 4/200Train SSM Loss: 0.0004291992335076555Val SSM Loss: 0.0003950286352644778
Models saved as models_best_ssm, ssm loss: 0.0003950286352644778
Epoch 5/200Train SSM Loss: 0.00040165119077395294Val SSM Loss: 0.000376052158470783
Models saved as models_best_ssm, ssm loss: 0.000376052158470783
Epoch 6/200Train SSM Loss: 0.00038117923500725074Val SSM Loss: 0.0003574186250463956
Models saved as models_best_ssm, ssm loss: 0.0003574186250463956
Epoch 7/200Train SSM Loss: 0.00036413035957353577Val SSM Loss: 0.00034457726805056963
Models saved as m